# Prompt Templates 与 Output Parsers

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_learning.llm import get_llm

llm = get_llm()

**术语：**
- **ChatPromptTemplate** = 提示模板，用 `{变量}` 占位，运行时会替换成真正的值
- **Output Parser** = 输出解析器，把 LLM 返回的文本处理成你想要的形式
- **StrOutputParser** = 最简单的解析器，只提取文本字符串
- **Chain（链）** = 用 `|` 把多个组件串起来，前一个的输出给后一个

```mermaid
graph LR
    A[用户输入] --> B[ChatPromptTemplate]
    B --> C[LLM]
    C --> D[StrOutputParser]
    D --> E[纯文本输出]
    style B fill:#E3F2FD,stroke:#1565C0
    style C fill:#FFF3E0,stroke:#E65100
    style D fill:#E8F5E9,stroke:#2E7D32
```

### 1. ChatPromptTemplate 带变量

In [2]:
prompt = ChatPromptTemplate.from_template(
    "你是一个{role}专家，请回答：{question}"
)
chain = prompt | llm | StrOutputParser()

print(chain.invoke({"role": "Python", "question": "什么是装饰器？"})[:200])

## 装饰器（Decorator）详解

**装饰器**是Python中一种强大的设计模式，它允许你在不修改原函数代码的情况下，动态地给函数或类添加额外的功能。

### 核心概念

装饰器本质上是一个**可调用对象**（通常是函数），它接收一个函数作为参数，并返回一个新的函数（或原函数的增强版本）。

### 基本语法

```python
# 装饰器的基本结构
def my_decorator


**术语：**
- **from_template** = 用字符串创建模板，`{role}` 和 `{question}` 是占位变量
- **invoke** = 执行链，传入一个字典（`{}`）给所有变量赋值

`ChatPromptTemplate.from_template()` 定义带变量的模板。`|` 将模板、LLM、解析器串联成一条链，调用 `invoke()` 时传入变量字典即可执行。

### 2. 结构化输出（Pydantic）

In [4]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class MovieReview(BaseModel):
    title: str = Field(description="电影名称")
    rating: float = Field(description="评分 1-10")
    summary: str = Field(description="一句话短评")

parser = PydanticOutputParser(pydantic_object=MovieReview)

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "提取结构化数据。{format_instructions}"),
    ("human", "{input}"),
]).partial(format_instructions=parser.get_format_instructions())

chain = review_prompt | llm | parser
review = chain.invoke({"input": "《黑客帝国》科幻片，基努·里维斯主演，评分9分。"})
print(f"电影: {review.title}")
print(f"评分: {review.rating}")
print(f"短评: {review.summary}")

电影: 黑客帝国
评分: 9.0
短评: 科幻经典，基努·里维斯主演，探讨现实与虚拟的边界。


**术语：**
- **Pydantic** = Python 的数据校验库，用来定义数据结构（字段名 + 类型 + 说明）
- **BaseModel / Field** = Pydantic 的基类和字段定义
- **Schema（模式）** = 数据的结构定义，告诉 LLM 要输出什么格式
- **format_instructions** = 自动生成的格式说明，告诉 LLM 按指定 schema 输出 JSON

```mermaid
graph LR
    A[原始文本] --> B[Prompt + 格式指令]
    B --> C[LLM]
    C --> D[JSON 文本]
    D --> E[PydanticParser]
    E --> F[结构化对象]
```